# Validação da qualidade dos dados

## Objetivo

Este notebook realiza a validação formal da qualidade dos dados disponibilizados na camada Silver do projeto.

As validações verificam:

- unicidade das chaves;
- preenchimento de campos obrigatórios;
- integridade referencial;
- validade dos domínios;
- consistência de datas;
- validade de valores numéricos;
- consistência dos cálculos financeiros;
- tratamento dos registros rejeitados.

Cada regra recebe um dos seguintes resultados:

- **APROVADO:** nenhuma violação identificada;
- **REPROVADO:** uma ou mais violações identificadas.

Os resultados são armazenados em uma tabela Delta para permitir rastreabilidade das execuções.

## Fluxo

Bronze → Silver → Validação de Qualidade → Gold

In [0]:
from pyspark.sql import functions as F
from datetime import datetime


# ---------------------------------------------------------
# CONFIGURAÇÃO
# ---------------------------------------------------------

catalogo_atual = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_silver = "varejo_silver"


print(f"Catálogo: {catalogo_atual}")
print(f"Schema avaliado: {schema_silver}")

In [0]:
# ---------------------------------------------------------
# CARREGAMENTO DAS TABELAS
# ---------------------------------------------------------

def carregar_silver(nome_tabela):
    """
    Carrega uma tabela da camada Silver.
    """

    return spark.table(
        f"{catalogo_atual}."
        f"{schema_silver}."
        f"{nome_tabela}"
    )


df_clientes = carregar_silver(
    "dim_cliente"
)

df_produtos = carregar_silver(
    "dim_produto"
)

df_vendedores = carregar_silver(
    "dim_vendedor"
)

df_fornecedores = carregar_silver(
    "dim_fornecedor"
)

df_vendas = carregar_silver(
    "fato_vendas"
)

df_compras = carregar_silver(
    "fato_compras"
)

df_estoque = carregar_silver(
    "fato_estoque"
)

df_receber = carregar_silver(
    "fato_contas_receber"
)

df_pagar = carregar_silver(
    "fato_contas_pagar"
)

df_metas = carregar_silver(
    "fato_metas_vendas"
)

df_quarentena_vendas = carregar_silver(
    "quarentena_vendas"
)

In [0]:
# ---------------------------------------------------------
# REGISTRO DOS TESTES
# ---------------------------------------------------------

resultados_qualidade = []


def registrar_teste(
    categoria,
    tabela,
    regra,
    quantidade_violacoes,
    criticidade="CRÍTICA"
):
    """
    Registra o resultado de uma regra de qualidade.
    """

    status = (
        "APROVADO"
        if quantidade_violacoes == 0
        else "REPROVADO"
    )

    resultados_qualidade.append(
        {
            "categoria": categoria,
            "tabela": tabela,
            "regra": regra,
            "criticidade": criticidade,
            "quantidade_violacoes":
                int(quantidade_violacoes),
            "status": status
        }
    )

In [0]:
# ---------------------------------------------------------
# UNICIDADE
# ---------------------------------------------------------

testes_unicidade = [
    (
        df_clientes,
        "dim_cliente",
        ["id_cliente"]
    ),
    (
        df_produtos,
        "dim_produto",
        ["id_produto"]
    ),
    (
        df_vendedores,
        "dim_vendedor",
        ["id_vendedor"]
    ),
    (
        df_fornecedores,
        "dim_fornecedor",
        ["id_fornecedor"]
    ),
    (
        df_vendas,
        "fato_vendas",
        ["id_item_venda"]
    ),
    (
        df_compras,
        "fato_compras",
        ["id_item_compra"]
    ),
    (
        df_receber,
        "fato_contas_receber",
        ["id_titulo_receber"]
    ),
    (
        df_pagar,
        "fato_contas_pagar",
        ["id_titulo_pagar"]
    ),
    (
        df_estoque,
        "fato_estoque",
        [
            "data_referencia",
            "id_produto"
        ]
    ),
    (
        df_metas,
        "fato_metas_vendas",
        [
            "mes_referencia",
            "id_vendedor"
        ]
    )
]


for dataframe, tabela, colunas in testes_unicidade:

    duplicados = (
        dataframe
        .groupBy(*colunas)
        .count()
        .filter(
            F.col("count") > 1
        )
        .count()
    )

    registrar_teste(
        categoria="Unicidade",
        tabela=tabela,
        regra=(
            "Chave única: "
            + ", ".join(colunas)
        ),
        quantidade_violacoes=duplicados
    )

In [0]:
# ---------------------------------------------------------
# COMPLETUDE
# ---------------------------------------------------------

campos_obrigatorios = [
    (
        df_clientes,
        "dim_cliente",
        [
            "id_cliente",
            "nome_cliente"
        ]
    ),
    (
        df_produtos,
        "dim_produto",
        [
            "id_produto",
            "nome_produto",
            "categoria"
        ]
    ),
    (
        df_vendedores,
        "dim_vendedor",
        [
            "id_vendedor",
            "nome_vendedor"
        ]
    ),
    (
        df_fornecedores,
        "dim_fornecedor",
        [
            "id_fornecedor",
            "nome_fornecedor"
        ]
    ),
    (
        df_vendas,
        "fato_vendas",
        [
            "id_item_venda",
            "id_venda",
            "data_venda",
            "id_cliente",
            "id_produto",
            "id_vendedor"
        ]
    ),
    (
        df_compras,
        "fato_compras",
        [
            "id_item_compra",
            "id_compra",
            "data_compra",
            "id_fornecedor",
            "id_produto"
        ]
    )
]

In [0]:
for dataframe, tabela, colunas in campos_obrigatorios:

    for coluna in colunas:

        violacoes = (
            dataframe
            .filter(
                F.col(coluna).isNull()
            )
            .count()
        )

        registrar_teste(
            categoria="Completude",
            tabela=tabela,
            regra=(
                f"{coluna} não pode ser nulo"
            ),
            quantidade_violacoes=violacoes
        )

In [0]:
vendas_clientes_invalidos = (

    df_vendas
    .select(
        "id_cliente"
    )
    .distinct()

    .join(
        df_clientes.select(
            "id_cliente"
        ),
        on="id_cliente",
        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_vendas",
    regra="Cliente da venda deve existir em dim_cliente",
    quantidade_violacoes=vendas_clientes_invalidos
)

In [0]:
vendas_produtos_invalidos = (

    df_vendas
    .select(
        "id_produto"
    )
    .distinct()

    .join(
        df_produtos.select(
            "id_produto"
        ),
        on="id_produto",
        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_vendas",
    regra="Produto da venda deve existir em dim_produto",
    quantidade_violacoes=vendas_produtos_invalidos
)

In [0]:
vendas_vendedores_invalidos = (

    df_vendas
    .select(
        "id_vendedor"
    )
    .distinct()

    .join(
        df_vendedores.select(
            "id_vendedor"
        ),
        on="id_vendedor",
        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_vendas",
    regra="Vendedor da venda deve existir em dim_vendedor",
    quantidade_violacoes=vendas_vendedores_invalidos
)

In [0]:
compras_produtos_invalidos = (

    df_compras
    .select(
        "id_produto"
    )
    .distinct()

    .join(
        df_produtos.select(
            "id_produto"
        ),
        on="id_produto",
        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_compras",
    regra="Produto da compra deve existir em dim_produto",
    quantidade_violacoes=compras_produtos_invalidos
)

In [0]:
compras_fornecedores_invalidos = (

    df_compras
    .select(
        "id_fornecedor"
    )
    .distinct()

    .join(
        df_fornecedores.select(
            "id_fornecedor"
        ),
        on="id_fornecedor",
        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_compras",
    regra="Fornecedor da compra deve existir em dim_fornecedor",
    quantidade_violacoes=compras_fornecedores_invalidos
)

In [0]:
estoque_produtos_invalidos = (

    df_estoque
    .select(
        "id_produto"
    )
    .distinct()

    .join(
        df_produtos.select(
            "id_produto"
        ),
        on="id_produto",
        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_estoque",
    regra="Produto do estoque deve existir em dim_produto",
    quantidade_violacoes=estoque_produtos_invalidos
)

In [0]:
metas_vendedores_invalidos = (

    df_metas
    .select(
        "id_vendedor"
    )
    .distinct()

    .join(
        df_vendedores.select(
            "id_vendedor"
        ),
        on="id_vendedor",
        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_metas_vendas",
    regra="Vendedor da meta deve existir em dim_vendedor",
    quantidade_violacoes=metas_vendedores_invalidos
)

In [0]:
violacoes_quantidade = (

    df_vendas
    .filter(
        F.col("quantidade") <= 0
    )
    .count()
)


registrar_teste(
    categoria="Validade",
    tabela="fato_vendas",
    regra="Quantidade vendida deve ser maior que zero",
    quantidade_violacoes=violacoes_quantidade
)

In [0]:
colunas_monetarias_vendas = [
    "preco_unitario",
    "valor_bruto",
    "valor_desconto",
    "valor_liquido",
    "custo_total"
]


for coluna in colunas_monetarias_vendas:

    violacoes = (
        df_vendas
        .filter(
            F.col(coluna) < 0
        )
        .count()
    )

    registrar_teste(
        categoria="Validade",
        tabela="fato_vendas",
        regra=(
            f"{coluna} não pode ser negativo"
        ),
        quantidade_violacoes=violacoes
    )

In [0]:
violacoes_valor_bruto = (

    df_vendas

    .withColumn(
        "valor_bruto_calculado",

        F.round(
            F.col("quantidade")
            * F.col("preco_unitario"),
            2
        )
    )

    .filter(
        F.abs(
            F.col("valor_bruto")
            - F.col(
                "valor_bruto_calculado"
            )
        ) > 0.02
    )

    .count()
)


registrar_teste(
    categoria="Consistência financeira",
    tabela="fato_vendas",
    regra=(
        "Valor bruto deve corresponder a "
        "quantidade × preço unitário"
    ),
    quantidade_violacoes=
        violacoes_valor_bruto
)

In [0]:
violacoes_valor_liquido = (

    df_vendas

    .filter(
        F.abs(
            F.col("valor_liquido")
            -
            (
                F.col("valor_bruto")
                - F.col("valor_desconto")
            )
        ) > 0.02
    )

    .count()
)


registrar_teste(
    categoria="Consistência financeira",
    tabela="fato_vendas",
    regra=(
        "Valor líquido deve corresponder a "
        "valor bruto - desconto"
    ),
    quantidade_violacoes=
        violacoes_valor_liquido
)

In [0]:
violacoes_lucro = (

    df_vendas

    .filter(
        F.abs(
            F.col("lucro_bruto")
            -
            (
                F.col("valor_liquido")
                - F.col("custo_total")
            )
        ) > 0.02
    )

    .count()
)


registrar_teste(
    categoria="Consistência financeira",
    tabela="fato_vendas",
    regra=(
        "Lucro bruto deve corresponder a "
        "valor líquido - custo"
    ),
    quantidade_violacoes=
        violacoes_lucro
)

In [0]:
violacoes_desconto = (

    df_vendas

    .filter(
        (F.col("percentual_desconto") < 0)
        |
        (F.col("percentual_desconto") > 1)
    )

    .count()
)


registrar_teste(
    categoria="Validade",
    tabela="fato_vendas",
    regra="Percentual de desconto deve estar entre 0 e 1",
    quantidade_violacoes=violacoes_desconto
)

In [0]:
violacoes_estoque_negativo = (

    df_estoque

    .filter(
        F.col(
            "quantidade_estoque"
        ) < 0
    )

    .count()
)


registrar_teste(
    categoria="Validade",
    tabela="fato_estoque",
    regra="Quantidade em estoque não pode ser negativa",
    quantidade_violacoes=
        violacoes_estoque_negativo
)

In [0]:
violacoes_limites_estoque = (

    df_estoque

    .filter(
        F.col("estoque_minimo")
        >
        F.col("estoque_maximo")
    )

    .count()
)


registrar_teste(
    categoria="Consistência",
    tabela="fato_estoque",
    regra="Estoque mínimo deve ser menor ou igual ao estoque máximo",
    quantidade_violacoes=
        violacoes_limites_estoque
)

In [0]:
status_receber_validos = [
    "A vencer",
    "Pago em dia",
    "Pago em atraso",
    "Vencido"
]


violacoes_status_receber = (

    df_receber

    .filter(
        ~F.col(
            "status_titulo"
        ).isin(
            status_receber_validos
        )
    )

    .count()
)


registrar_teste(
    categoria="Domínio",
    tabela="fato_contas_receber",
    regra="Status financeiro deve pertencer ao domínio permitido",
    quantidade_violacoes=
        violacoes_status_receber
)

In [0]:
violacoes_vencimento = (

    df_receber

    .filter(
        F.col("data_vencimento")
        <
        F.col("data_emissao")
    )

    .count()
)


registrar_teste(
    categoria="Consistência temporal",
    tabela="fato_contas_receber",
    regra="Vencimento não pode ocorrer antes da emissão",
    quantidade_violacoes=
        violacoes_vencimento
)

In [0]:
violacoes_pagamento = (

    df_receber

    .filter(
        F.col("data_pagamento").isNotNull()
        &
        (
            F.col("data_pagamento")
            <
            F.col("data_emissao")
        )
    )

    .count()
)


registrar_teste(
    categoria="Consistência temporal",
    tabela="fato_contas_receber",
    regra="Pagamento não pode ocorrer antes da emissão",
    quantidade_violacoes=
        violacoes_pagamento
)

In [0]:
violacoes_receber_negativo = (

    df_receber

    .filter(
        F.col("valor_titulo") <= 0
    )

    .count()
)


registrar_teste(
    categoria="Validade",
    tabela="fato_contas_receber",
    regra="Valor do título deve ser maior que zero",
    quantidade_violacoes=
        violacoes_receber_negativo
)

In [0]:
portes_validos = [
    "Pequeno",
    "Médio",
    "Grande"
]


violacoes_porte = (

    df_clientes

    .filter(
        ~F.col(
            "porte_cliente"
        ).isin(
            portes_validos
        )
    )

    .count()
)


registrar_teste(
    categoria="Domínio",
    tabela="dim_cliente",
    regra="Porte do cliente deve pertencer ao domínio permitido",
    quantidade_violacoes=
        violacoes_porte
)

In [0]:
situacoes_cliente_validas = [
    "Ativo",
    "Inativo"
]


violacoes_situacao_cliente = (

    df_clientes

    .filter(
        ~F.col(
            "situacao_cliente"
        ).isin(
            situacoes_cliente_validas
        )
    )

    .count()
)


registrar_teste(
    categoria="Domínio",
    tabela="dim_cliente",
    regra="Situação do cliente deve ser Ativo ou Inativo",
    quantidade_violacoes=
        violacoes_situacao_cliente
)

In [0]:
categorias_validas = [
    "Alimentos",
    "Bebidas",
    "Higiene",
    "Limpeza",
    "Papelaria",
    "Utilidades"
]


violacoes_categoria = (

    df_produtos

    .filter(
        ~F.col(
            "categoria"
        ).isin(
            categorias_validas
        )
    )

    .count()
)


registrar_teste(
    categoria="Domínio",
    tabela="dim_produto",
    regra="Categoria do produto deve pertencer ao domínio permitido",
    quantidade_violacoes=
        violacoes_categoria
)

In [0]:
titulos_sem_venda = (

    df_receber

    .select(
        "id_venda"
    )
    .distinct()

    .join(
        df_vendas
        .select(
            "id_venda"
        )
        .distinct(),

        on="id_venda",

        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_contas_receber",
    regra="Título a receber deve possuir uma venda correspondente",
    quantidade_violacoes=
        titulos_sem_venda
)

In [0]:
titulos_sem_compra = (

    df_pagar

    .select(
        "id_compra"
    )
    .distinct()

    .join(
        df_compras
        .select(
            "id_compra"
        )
        .distinct(),

        on="id_compra",

        how="left_anti"
    )

    .count()
)


registrar_teste(
    categoria="Integridade referencial",
    tabela="fato_contas_pagar",
    regra="Título a pagar deve possuir uma compra correspondente",
    quantidade_violacoes=
        titulos_sem_compra
)

In [0]:
# ---------------------------------------------------------
# RESULTADOS CONSOLIDADOS
# ---------------------------------------------------------

df_resultados_qualidade = (

    spark.createDataFrame(
        resultados_qualidade
    )

    .withColumn(
        "data_execucao",
        F.current_timestamp()
    )

    .select(
        "categoria",
        "tabela",
        "regra",
        "criticidade",
        "quantidade_violacoes",
        "status",
        "data_execucao"
    )
)

In [0]:
display(
    df_resultados_qualidade
    .orderBy(
        "status",
        "categoria",
        "tabela"
    )
)

In [0]:
df_resumo_qualidade = (

    df_resultados_qualidade

    .groupBy(
        "status"
    )

    .agg(
        F.count("*").alias(
            "quantidade_testes"
        )
    )
)


display(
    df_resumo_qualidade
)

In [0]:
total_testes = (
    df_resultados_qualidade.count()
)


testes_aprovados = (

    df_resultados_qualidade

    .filter(
        F.col("status")
        == "APROVADO"
    )

    .count()
)


percentual_aprovacao = (
    testes_aprovados
    / total_testes
    * 100
)


print(
    f"Testes executados: {total_testes}"
)

print(
    f"Testes aprovados: {testes_aprovados}"
)

print(
    f"Taxa de aprovação: "
    f"{percentual_aprovacao:.2f}%"
)

In [0]:
nome_tabela_qualidade = (
    f"{catalogo_atual}."
    f"{schema_silver}."
    f"resultado_qualidade"
)


(
    df_resultados_qualidade

    .write

    .format("delta")

    .mode("overwrite")

    .option(
        "overwriteSchema",
        "true"
    )

    .saveAsTable(
        nome_tabela_qualidade
    )
)


print(
    f"Resultado gravado em: "
    f"{nome_tabela_qualidade}"
)

In [0]:
display(
    df_quarentena_vendas

    .groupBy(
        "motivo_rejeicao"
    )

    .agg(
        F.count("*").alias(
            "quantidade_registros"
        )
    )

    .orderBy(
        F.desc(
            "quantidade_registros"
        )
    )
)

In [0]:
titulos_sem_venda_detalhes = (

    df_receber.alias("r")

    .join(
        df_vendas
        .select("id_venda")
        .distinct()
        .alias("v"),

        on="id_venda",

        how="left_anti"
    )

    .select(
        "id_titulo_receber",
        "id_venda",
        "id_cliente",
        "data_emissao",
        "data_vencimento",
        "valor_titulo",
        "status_titulo"
    )
)


display(
    titulos_sem_venda_detalhes
)

In [0]:
ids_vendas_sem_correspondencia = [

    linha["id_venda"]

    for linha in (
        titulos_sem_venda_detalhes
        .select("id_venda")
        .distinct()
        .collect()
    )
]


display(

    df_quarentena_vendas

    .filter(
        F.col("id_venda")
        .isin(ids_vendas_sem_correspondencia)
    )

    .select(
        "id_item_venda",
        "id_venda",
        "id_cliente",
        "id_produto",
        "id_vendedor",
        "quantidade",
        "motivo_rejeicao"
    )
)

In [0]:
testes_criticos_reprovados = (

    df_resultados_qualidade

    .filter(
        (F.col("status") == "REPROVADO")
        &
        (F.col("criticidade") == "CRÍTICA")
    )

    .count()
)


if testes_criticos_reprovados > 0:

    raise Exception(
        f"Foram identificados "
        f"{testes_criticos_reprovados} "
        f"testes críticos reprovados. "
        f"A camada Gold não deve ser processada."
    )


print(
    "Validação concluída com sucesso. "
    "Os dados estão aptos para a camada Gold."
)